# ⚛️ Quantum Tic-Tac-Toe — Solution Notebook
**Qiskit Fall Fest 2026 — IIIT Kottayam | Track 1: Quantum Games**

---

## Overview
This notebook demonstrates the **quantum mechanics core** of our Quantum Tic-Tac-Toe implementation.

We explore:
1. **Superposition** — Hadamard gate puts a qubit in 50/50 state
2. **Entanglement** — CNOT creates Bell states linking two qubits
3. **Measurement Collapse** — Measuring collapses the superposition
4. **A full game simulation** — Quantum vs Classical comparison

Based on Allan Goff's [Quantum Tic-Tac-Toe paper](https://arxiv.org/abs/quant-ph/0202151).

## 0. Setup & Installation

In [ ]:
# Install packages if needed (uncomment in a fresh environment)
# !pip install qiskit qiskit-aer matplotlib numpy streamlit

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram

print("✅ All packages imported successfully!")
import qiskit
print(f"   Qiskit version: {qiskit.__version__}")

---
## 1. Quantum Principle #1 — Superposition (H Gate)

**Game mechanic:** When a player places a mark in TWO cells simultaneously,
the mark is in a **superposition** — it could be in either cell.

The Hadamard (H) gate transforms:
$$|0\rangle \xrightarrow{H} \frac{|0\rangle + |1\rangle}{\sqrt{2}}$$

- State $|0\rangle$ = mark is NOT in this cell
- State $|1\rangle$ = mark IS in this cell
- After H: 50% chance of either!

In [ ]:
# --- Demonstrating Superposition ---
# 1 qubit = 1 cell on the board
# We apply H and then measure many times to show the 50/50 distribution

qc_superposition = QuantumCircuit(1, 1)

# Start in |0⟩ (no mark)
# Apply H gate → superposition (mark could be here OR not)
qc_superposition.h(0)

# Measure to see the result
qc_superposition.measure(0, 0)

print("Superposition Circuit:")
print(qc_superposition.draw(output='text'))

In [ ]:
# Run 1000 shots to show the 50/50 distribution
sampler = StatevectorSampler()
job = sampler.run([qc_superposition], shots=1000)
result = job.result()[0]
counts = result.data.c.get_counts()

print(f"Results from 1000 measurements:")
print(f"  |0⟩ (no mark): {counts.get('0', 0)} times ({counts.get('0', 0)/10:.1f}%)")
print(f"  |1⟩ (mark here): {counts.get('1', 0)} times ({counts.get('1', 0)/10:.1f}%)")
print()
print("✅ Roughly 50/50 — true quantum randomness!")

# Plot
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['|0⟩ (No Mark)', '|1⟩ (Mark Here)'],
       [counts.get('0', 0), counts.get('1', 0)],
       color=['#4A90D9', '#D94A4A'])
ax.set_title('Superposition: 50/50 Measurement Outcomes')
ax.set_ylabel('Count (out of 1000 shots)')
plt.tight_layout()
plt.show()

---
## 2. Quantum Principle #2 — Entanglement (CNOT Gate)

**Game mechanic:** When a player places a quantum mark in cells A and B,
those cells become **entangled** — measuring one instantly determines the other.

We create a **Bell state** using H + CNOT:
$$|00\rangle \xrightarrow{H\otimes I} \frac{|0\rangle+|1\rangle}{\sqrt{2}}|0\rangle \xrightarrow{CNOT} \frac{|00\rangle + |11\rangle}{\sqrt{2}}$$

This means:
- If qubit A = 0 → qubit B = 0 (mark not in A, not in B... but somewhere!)
- If qubit A = 1 → qubit B = 1 (both register the mark)

They are **always correlated** — no matter how far apart they are!

In [ ]:
# --- Demonstrating Entanglement ---
# 2 qubits = 2 cells on the board
# A quantum mark placed in cells 0 and 3 (for example)

qc_entangled = QuantumCircuit(2, 2)

# Step 1: H gate on qubit 0 (cell_a) — creates superposition
qc_entangled.h(0)

# Step 2: CNOT from qubit 0 (control) to qubit 1 (target) (cell_b)
# This ENTANGLES the two qubits → Bell state (|00⟩ + |11⟩)/√2
qc_entangled.cx(0, 1)

# Measure both qubits
qc_entangled.measure([0, 1], [0, 1])

print("Entanglement Circuit (H + CNOT = Bell State):")
print(qc_entangled.draw(output='text'))

In [ ]:
# Run 1000 shots — notice we ONLY get '00' or '11', never '01' or '10'!
job = sampler.run([qc_entangled], shots=1000)
result = job.result()[0]
counts = result.data.c.get_counts()

print("Results from 1000 measurements of entangled pair:")
for bitstring, count in sorted(counts.items()):
    print(f"  |{bitstring}⟩ : {count} times ({count/10:.1f}%)")

print()
print("✅ KEY INSIGHT: We ONLY see '00' or '11' — NEVER '01' or '10'!")
print("   The qubits are perfectly correlated — this is ENTANGLEMENT.")
print("   In the game: both cells always share the same fate.")

# Plot
fig, ax = plt.subplots(figsize=(6, 3))
all_states = {'00': 0, '01': 0, '10': 0, '11': 0}
all_states.update(counts)
colors = ['#4A90D9' if s in ['00', '11'] else '#555' for s in all_states]
ax.bar(list(all_states.keys()), list(all_states.values()), color=colors)
ax.set_title('Entangled Pair: Only |00⟩ and |11⟩ Outcomes')
ax.set_xlabel('Measurement Outcome (qubit1 qubit0)')
ax.set_ylabel('Count (out of 1000 shots)')
plt.tight_layout()
plt.show()

---
## 3. Quantum Principle #3 — Measurement Collapse

**Game mechanic:** When an **entanglement cycle** forms, the board collapses.
All superpositions resolve into definite classical marks.

This is the game's most dramatic moment — like Schrödinger opening the box!

We'll simulate a 3-cell cycle: cells 0, 1, 2 are entangled in a loop.

In [ ]:
# --- Simulating a 3-cell entanglement cycle ---
# Scenario:
#   Move 1 (X): places X₁ in cells 0 and 1  → H(q0), CNOT(q0→q1)
#   Move 2 (O): places O₂ in cells 1 and 2  → H(q1), CNOT(q1→q2)
#   Move 3 (X): places X₃ in cells 2 and 0  → H(q2), CNOT(q2→q0) ← CYCLE!
#
# Cells 0→1→2→0 form a closed loop → COLLAPSE is triggered!

qc_cycle = QuantumCircuit(3, 3)

# Move 1: X₁ in cells 0 and 1
qc_cycle.barrier(label="Move 1: X in (0,1)")
qc_cycle.h(0)
qc_cycle.cx(0, 1)

# Move 2: O₂ in cells 1 and 2
qc_cycle.barrier(label="Move 2: O in (1,2)")
qc_cycle.h(1)
qc_cycle.cx(1, 2)

# Move 3: X₃ in cells 2 and 0 — CREATES CYCLE 0→1→2→0
qc_cycle.barrier(label="Move 3: X in (2,0) ← CYCLE!")
qc_cycle.h(2)
qc_cycle.cx(2, 0)

# COLLAPSE: Measure all 3 qubits
qc_cycle.barrier(label="COLLAPSE")
qc_cycle.measure([0, 1, 2], [0, 1, 2])

print("3-cell Entanglement Cycle Circuit:")
print(qc_cycle.draw(output='text'))

In [ ]:
# Run 1000 shots to see the collapse probability distribution
job = sampler.run([qc_cycle], shots=1000)
result = job.result()[0]
counts = result.data.c.get_counts()

print("Collapse outcomes for 3-cell cycle (1000 simulations):")
print("Bitstring format: qubit2 qubit1 qubit0 (little-endian)")
print()
for bitstring, count in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  |{bitstring}⟩ : {count} times ({count/10:.1f}%)")

print()
print("✅ The collapse is PROBABILISTIC — different resolutions are possible!")
print("   In the game, the opponent CHOOSES which resolution to apply.")

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(list(counts.keys()), list(counts.values()), color='#e94560')
ax.set_title('3-Cell Cycle Collapse: Probability Distribution')
ax.set_xlabel('Measurement Outcome (cell2 cell1 cell0)')
ax.set_ylabel('Count (out of 1000 shots)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 4. Full Quantum Circuit: A Complete Game

Here we build the **9-qubit circuit** (one qubit per board cell) and
simulate a short game sequence using our `QuantumEngine`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

from game_logic import GameState
from quantum_engine import QuantumEngine

# Initialize fresh game and quantum engine
game = GameState()
engine = QuantumEngine()

print("✅ Game and Quantum Engine initialized!")
print(f"   Board: 9 cells (0-8)")
print(f"   Circuit: 9 qubits, 9 classical bits")
print()
print("Board layout:")
print("  0 | 1 | 2")
print("  ---------")
print("  3 | 4 | 5")
print("  ---------")
print("  6 | 7 | 8")

In [ ]:
# --- PLAY A SHORT GAME SEQUENCE ---

def print_board(game):
    """Print the board state in a readable format."""
    print()
    for row in range(3):
        row_str = []
        for col in range(3):
            cell = row * 3 + col
            display = game.get_cell_display(cell)
            row_str.append(display.center(12) if display else f"  {cell}  ".center(12))
        print(" | ".join(row_str))
        if row < 2:
            print("-" * 42)
    print()

print("Initial empty board:")
print_board(game)

In [ ]:
# Move 1: X places X₁ in cells 0 and 4 (center)
print("Move 1: X places quantum mark in cells 0 (Top-Left) and 4 (Center)")
cycle = game.place_quantum_mark(0, 4)
engine.add_quantum_mark(0, 4, 1)
print(f"  Cycle detected: {cycle}")
print_board(game)
print("  Quantum state: X₁ exists in BOTH cell 0 AND cell 4 simultaneously!")

In [ ]:
# Move 2: O places O₂ in cells 4 and 8 (diagonal)
print("Move 2: O places quantum mark in cells 4 (Center) and 8 (Bottom-Right)")
cycle = game.place_quantum_mark(4, 8)
engine.add_quantum_mark(4, 8, 2)
print(f"  Cycle detected: {cycle}")
print_board(game)
print("  Now cell 4 has BOTH X₁ and O₂ floating in it!")

In [ ]:
# Move 3: X places X₃ in cells 8 and 0 — THIS CREATES A CYCLE!
# Cycle: 0 → 4 → 8 → 0 (via marks X₁, O₂, X₃)
print("Move 3: X places quantum mark in cells 8 and 0")
print("        This creates a CYCLE: 0 → 4 → 8 → 0")
cycle = game.place_quantum_mark(8, 0)
engine.add_quantum_mark(8, 0, 3)
print(f"  ⚡ Cycle detected! Cells: {cycle}")
print_board(game)
print(f"  Collapse options available: {len(game.collapse_options)}")
for i, option in enumerate(game.collapse_options):
    print(f"    Option {i+1}: {option}")

In [ ]:
# COLLAPSE: Opponent (O) chooses Option 0
print("Collapse: Player O chooses Option 1")

# Measure qubits using the quantum engine
cells_in_cycle = cycle if cycle else [0, 4, 8]
try:
    outcomes = engine.measure_and_collapse(cells_in_cycle)
    print(f"  Qiskit measurement outcomes: {outcomes}")
except Exception as e:
    print(f"  (Measurement note: {e})")

# Apply collapse via game logic
if game.collapse_options:
    game.apply_collapse(0)

print()
print("Board after collapse:")
print_board(game)
print("  Classical marks are now DEFINITE — no more superposition!")
print(f"  Classical cells: {game.classical_cells}")

In [ ]:
# Show the Quantum Circuit built so far
print("The Quantum Circuit built during this game:")
print(engine.get_circuit_diagram())

---
## 5. Classical vs Quantum Comparison

We run **100 simulated games** between:
- **Random vs Random** (pure classical chance)
- **Smart vs Smart** (strategic classical players)

The key insight: In **classical** Tic-Tac-Toe, two perfect players **always draw**.
In **Quantum** Tic-Tac-Toe, collapse randomness breaks this — wins and losses occur!

In [ ]:
from classical_baseline import simulate_games

NUM_GAMES = 200

print(f"Simulating {NUM_GAMES} games for each scenario...")
print()

# Scenario 1: Random vs Random
stats_rr = simulate_games(NUM_GAMES, 'random', 'random')
print("Random vs Random:")
print(f"  X wins: {stats_rr['x_wins']} ({stats_rr['x_wins']/NUM_GAMES*100:.1f}%)")
print(f"  O wins: {stats_rr['o_wins']} ({stats_rr['o_wins']/NUM_GAMES*100:.1f}%)")
print(f"  Draws:  {stats_rr['draws']}  ({stats_rr['draws']/NUM_GAMES*100:.1f}%)")
print()

# Scenario 2: Smart vs Smart
stats_ss = simulate_games(NUM_GAMES, 'smart', 'smart')
print("Smart vs Smart:")
print(f"  X wins: {stats_ss['x_wins']} ({stats_ss['x_wins']/NUM_GAMES*100:.1f}%)")
print(f"  O wins: {stats_ss['o_wins']} ({stats_ss['o_wins']/NUM_GAMES*100:.1f}%)")
print(f"  Draws:  {stats_ss['draws']}  ({stats_ss['draws']/NUM_GAMES*100:.1f}%)")

In [ ]:
# --- Plot the comparison ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, stats, title in [
    (axes[0], stats_rr, f'Random vs Random\n({NUM_GAMES} games)'),
    (axes[1], stats_ss, f'Smart vs Smart\n({NUM_GAMES} games)')
]:
    labels = ['X Wins', 'O Wins', 'Draws']
    values = [stats['x_wins'], stats['o_wins'], stats['draws']]
    colors = ['#4A90D9', '#D94A4A', '#888888']
    bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('Number of Games')
    ax.set_ylim(0, NUM_GAMES)
    # Add percentage labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{val/NUM_GAMES*100:.1f}%', ha='center', va='bottom', fontsize=11)

fig.suptitle('Quantum Tic-Tac-Toe: Win/Loss/Draw Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print("✅ Chart saved as results_comparison.png")

---
## 6. Key Takeaways

| Concept | Classical Game | Quantum Game |
|---|---|---|  
| **Mark placement** | One mark, one cell | One mark, two cells (superposition) |
| **Cell state** | Always definite | Quantum marks until collapse |
| **Strategy depth** | Fixed board | Entanglement web creates extra layer |
| **Collapse** | N/A | Opponent chooses resolution |
| **Determinism** | Perfect play = Draw | Even smart players get wins/losses |
| **Quantum gate** | N/A | H (superposition) + CNOT (entanglement) |

### How to Run the Full Game
```bash
# From the project root:
pip install -r requirements.txt
streamlit run src/app.py
```
Then open your browser at `http://localhost:8501`!